<h1 style="font-size: 1.6rem; font-weight: bold">ITO5163: Introduction to Cybersecurity</h1>
<h1 style="font-size: 1.6rem; font-weight: bold">Module 4: Key Establishment</h1>
<p style="margin-top: 5px; margin-bottom: 5px;">Monash University Australia</p>
<p style="margin-top: 5px; margin-bottom: 5px;">Jupyter Notebook by: Tristan Sim Yook Min</p>
References: Information Source from Monash Faculty of Information Technology

---

### **Symmetric Key Distribution Using Symmetric Encryption**

Symmetric encryption is **fast** and can encrypt large amounts of data quickly. Its big weakness is **key distribution**: both parties need the **same key**, so two parties who have **never communicated before** need a way to share it securely without anyone else getting it.

#### **Ways to Share a Key**

- **Option 1**: A chooses a key and **physically delivers** it to B
- **Option 2**: A third party C, **trusted by both**, chooses a key and physically delivers it to A and B
- **Option 3**: If A and B **already share a key**, one of them sends a new key encrypted with the old one
- **Option 4**: If A and B each have an **encrypted channel to C**, C sends a key to both over those channels

| Option | Practical? | Problem |
|--------|------------|---------|
| 1 and 2: physical delivery | For a few very sensitive links | Impossible for users far apart, or for machines talking to machines |
| 3: use the old key | Yes, once a key exists | If the old key is ever stolen, every key after it is exposed |
| 4: trusted third party | **Yes, the most practical** | The third party must be trusted, and becomes a target |

#### **The Scale Problem**

If every pair of $n$ users needs its own key, each user needs $n - 1$ keys, and the network needs $\frac{n(n-1)}{2}$ keys in total. Option 4 is the basis of a **Key Distribution Center (KDC)**. Each user shares **just one key with the KDC**, and the KDC hands out keys for everything else.

| Users $n$ | Keys Needed, Every Pair | Master Keys Needed, with a KDC |
|-----------|-------------------------|--------------------------------|
| 10        | 45                      | 10                             |
| 100       | 4,950                   | 100                            |
| 1,000     | 499,500                 | 1,000                          |

#### **Key Hierarchy**

A KDC uses **two levels of keys**:

- **Master key**: shared **long-term** between the KDC and one user
  - Used **only** to encrypt messages between the KDC and that user
  - Still has to be delivered securely, e.g. physically, but there are far fewer of them
- **Session key**: a **temporary** key issued by the KDC for **one conversation**
  - Used to encrypt the actual data between two users
  - **Thrown away** when the session ends

#### **Key Distribution Scenario**

User A wants to talk to User B. $K_a$ and $K_b$ are their master keys with the KDC, and $K_s$ is the new session key.

```mermaid
sequenceDiagram
    participant A as User A
    participant K as KDC
    participant B as User B
    A->>K: 1. IDa, IDb, N1
    K->>A: 2. E(Ka, [Ks, IDa, IDb, N1, E(Kb, [Ks, IDa])])
    A->>B: 3. E(Kb, [Ks, IDa])
    B->>A: 4. E(Ks, N2)
    A->>B: 5. E(Ks, f(N2))
    Note over A,B: Steps 1-3: key distribution. Steps 4-5: authentication
```

| Step | Message | Purpose |
|------|---------|---------|
| 1 | $ID_A \parallel ID_B \parallel N_1$ | A asks the KDC for a session key to talk to B. $N_1$ is a **nonce**: a random value used once |
| 2 | $E(K_a, [K_s \parallel ID_A \parallel ID_B \parallel N_1 \parallel E(K_b, [K_s \parallel ID_A])])$ | The KDC sends $K_s$, encrypted so only A can read it. Repeating the request lets A check it **wasn't changed**, and the matching $N_1$ shows the reply is **new**, not an old one being replayed. It also includes a "ticket" for B |
| 3 | $E(K_b, [K_s \parallel ID_A])$ | A forwards the ticket. Only B can read it, so B knows it **came from the KDC** and that it's talking to **A** |
| 4 | $E(K_s, N_2)$ | B sends a new nonce, encrypted with the session key |
| 5 | $E(K_s, f(N_2))$ | A replies with a transformed nonce, e.g. $N_2 + 1$, proving A **really has $K_s$** and that step 3 **wasn't a replay** |

- **Steps 1 to 3** distribute the key. **Steps 4 and 5** are for authentication
- **Kerberos**, used for logins on Windows networks, is a real-world system built on this idea

#### **Automatic Key Distribution**

In the scenario above, the **user** has to pass messages between the KDC and the other user. In **automatic key distribution**, a **Session Security Module (SSM)** on each host handles this automatically, for connection-oriented protocols such as **TCP**.

```mermaid
sequenceDiagram
    participant A as Host A
    participant SA as SSM A
    participant K as KDC
    participant SB as SSM B
    participant B as Host B
    A->>SA: 1. Connection request
    SA->>K: 2. Request permission (encrypted with SSM A and KDC key)
    K->>SA: 3. Session key
    K->>SB: 3. Session key
    SA->>SB: 4. Connection set up, data encrypted with session key
    SB->>B: Data delivered
```

- **Steps**:
  - Step 1: Host A asks to connect, and the request goes to **its own SSM**
  - Step 2: The SSM asks the KDC for permission, encrypted with a key shared only by that SSM and the KDC
  - Step 3: If approved, the KDC sends the session key to **both** SSMs
  - Step 4: The SSMs set up the connection, and all data is encrypted with the session key
  - Users don't need to do anything themselves

#### **Things That Affect a KDC System**

**Hierarchical key control.** One KDC for a large network can get **overloaded**. Instead, there can be **several local KDCs**, each serving one area, with a **global KDC** linking them.

```mermaid
flowchart TB
    G["Global KDC"] --- L1["Local KDC 1"] & L2["Local KDC 2"]
    L1 --- A1["Users in area 1"]
    L2 --- B1["Users in area 2"]
```

- Users in the same area use their **local KDC**
- For users in different areas, the local KDCs work together through the **global KDC**
- **Less master key distribution** work, since each KDC only handles its own area
- A **faulty or compromised KDC** only affects **its own area**

**Session key lifetime.**

- **Short lifetime**: more secure, since less ciphertext is encrypted with each key for an attacker to analyse
- **Long lifetime**: less network overhead from fetching new keys
- Administrators must find the **right balance**

**Centralised vs decentralised.** A single KDC is easy to trust and manage, but it's also an **attractive target** for attackers. In a **decentralised** scheme, users share master keys **directly with each other**, with no KDC. This doesn't scale well to large networks, but works for **small local networks**.

```mermaid
sequenceDiagram
    participant A as User A
    participant B as User B
    Note over A,B: A and B already share a master key Km
    A->>B: 1. IDa, N1
    B->>A: 2. E(Km, [Ks, IDa, IDb, f(N1), N2])
    A->>B: 3. E(Ks, f(N2))
```

- **Decentralised Steps**:
  - Step 1: A sends its identity and a nonce $N_1$
  - Step 2: B picks a session key $K_s$ and sends it encrypted with their shared master key, with $f(N_1)$ to prove freshness, and a new nonce $N_2$
  - Step 3: A replies with $f(N_2)$, encrypted with the new session key
  - Each user needs a master key for **every** other user, $n - 1$ in total, which is why it only suits small networks

**Key separation.** A system holds many types of keys, and they must be **kept apart** so a key isn't used for the wrong purpose or leaked by mistake. One method is to **tag** each key with extra bits that say what it's allowed to do. These are called **key tags** or **control vectors**. Examples of key types:

- Master keys and session keys
- Keys for encrypting **data** sent over the network
- Keys for encrypting **PINs**, used at point-of-sale terminals and for electronic fund transfers
- Keys for encrypting **files** stored in publicly accessible places

<br>

***

### **Symmetric Key Distribution Using Asymmetric Encryption**

Asymmetric encryption is **too slow** for large amounts of data, but it's ideal for encrypting **small** things, like a **symmetric key**. Once both parties have the symmetric key, they switch to fast symmetric encryption for the actual data. This lesson covers three ways to do this.

#### **Simple Secret Key Distribution**

The simplest approach, proposed by **Merkle**:

```mermaid
sequenceDiagram
    participant A as User A
    participant B as User B
    Note over A: Generates a new key pair PUa, PRa
    A->>B: 1. PUa, IDa
    Note over B: Generates session key Ks
    B->>A: 2. E(PUa, Ks)
    Note over A: 3. Decrypts with PRa to get Ks
    Note over A,B: 4. Both use Ks, and throw away PUa and PRa
```

- **Steps**:
  - Step 1: A creates a **new** key pair $[PU_a, PR_a]$ and sends $PU_a$ and its identity $ID_A$ to B
  - Step 2: B creates a session key $K_s$, encrypts it with $PU_a$, and sends $E(PU_a, K_s)$
  - Step 3: A decrypts with $PR_a$ to get $K_s$
  - Step 4: Both use $K_s$, then **throw away** $PU_a$ and $PR_a$

- **Strengths**:
  - Very simple
  - **No keys exist before or after** the exchange, so there are no long-term keys to steal
  - Safe against **eavesdropping** (passive attacks), since an eavesdropper sees $E(PU_a, K_s)$ but can't decrypt it

- **Weakness: Man-in-the-Middle**: there's no way to check that $PU_a$ really came from A

```mermaid
sequenceDiagram
    participant A as User A
    participant X as Attacker
    participant B as User B
    A->>X: 1. PUa, IDa
    Note over X: Keeps PUa, makes own key pair PUe, PRe
    X->>B: PUe, IDa (pretends to be A)
    B->>X: 2. E(PUe, Ks)
    Note over X: Decrypts with PRe, learns Ks
    X->>A: E(PUa, Ks)
    Note over A,B: A and B use Ks, unaware the attacker has it too
```

- The attacker **swaps** A's public key for their own, learns $K_s$, and passes it on, so A and B notice nothing
- The attacker can then read everything A and B send each other
- So this scheme is only suitable when **eavesdropping** is the only threat

#### **Secret Key Distribution with Confidentiality and Authentication**

This scheme protects against **both passive and active attacks**, including man-in-the-middle. It requires A and B to have **already exchanged public keys** securely beforehand.

```mermaid
sequenceDiagram
    participant A as User A
    participant B as User B
    Note over A,B: A and B already have each other's public keys
    A->>B: 1. E(PUb, [N1, IDa])
    B->>A: 2. E(PUa, [N1, N2])
    A->>B: 3. E(PUb, N2)
    A->>B: 4. E(PUb, E(PRa, Ks))
    Note over B: 5. Decrypts with PRb, then checks with PUa to get Ks
    Note over A,B: 6. Both use Ks
```

| Step | Message | Purpose |
|------|---------|---------|
| 1 | $E(PU_b, [N_1 \parallel ID_A])$ | A starts, with a nonce $N_1$. Only B can read it |
| 2 | $E(PU_a, [N_1 \parallel N_2])$ | B returns $N_1$, which **proves it's really B**, since only B could decrypt step 1. B adds a new nonce $N_2$ |
| 3 | $E(PU_b, N_2)$ | A returns $N_2$, which **proves it's really A** |
| 4 | $E(PU_b, E(PR_a, K_s))$ | A sends the session key, **signed** with A's private key and **encrypted** with B's public key |
| 5 | B decrypts with $PR_b$, then uses $PU_a$ | Using $PR_b$ means only B can read it (**confidentiality**). Using $PU_a$ proves A sent it (**authentication**) |
| 6 | | Both now use $K_s$ |

- **Why It Works**:
  - The nonces prove that **both sides are live** and not a replay
  - Encrypting with $PU_b$ means only B can read $K_s$
  - Signing with $PR_a$ means only A could have sent $K_s$

#### **Hybrid Scheme**

Used on **IBM mainframes**, this combines a **KDC** (lesson 11.1) with **public-key encryption**:

- The KDC still uses a **master key** shared with each user to send out **session keys**
- **Public-key encryption** is used to send each user's **master key** to the KDC, and to replace it when needed

| Key | How It's Sent | How Often |
|-----|---------------|-----------|
| Master key (user and KDC) | Public-key encryption | **Occasionally** |
| Session keys | Symmetric encryption with the master key | **Frequently** |

- **Advantages**:
  - **Performance**: for systems with many transactions, the fast symmetric KDC handles all the frequent session key requests. Slow public-key operations are only needed for the occasional master key change. Changing master keys regularly also gives an attacker less ciphertext to analyse
  - **Backward compatibility**: it can be added on top of an **existing KDC system** with minimal disruption or software changes

#### **Comparison**

| Scheme | Prior Setup Needed | Stops Eavesdropping | Stops Man-in-the-Middle |
|--------|--------------------|---------------------|-------------------------|
| Simple (Merkle) | None | Yes | **No** |
| Confidentiality + authentication | Public keys already exchanged | Yes | Yes |
| Hybrid (KDC + public key) | A KDC, plus public keys for master keys | Yes | Yes |

The second scheme depends on having the **right public keys** to begin with. That's the problem the next topic, **public key distribution and certificates**, solves.

<br>

***

### **Distribution of Public Keys**

Public keys can be shared **openly**, but that raises a different problem: how do you know a public key **really belongs** to the person it claims to? If an attacker can pass off their own key as someone else's, they can read messages meant for that person. The user has to **trust the source** of the key.

There are four main approaches, each more secure than the last:

| Approach | How Keys Are Shared | Main Weakness |
|----------|---------------------|---------------|
| **Public announcement** | Users broadcast their own keys | Anyone can **forge** a key in someone else's name |
| **Publicly available directory** | A trusted directory stores keys | The directory is a **target**: if it's compromised, every key is suspect |
| **Public-key authority** | An online authority sends signed keys on request | A **bottleneck**: it must be contacted every time |
| **Public-key certificates** | A CA signs certificates once, users swap them directly | Relies on trusting the CA, and handling expiry and revocation |

#### **1. Public Announcement**

Each user **broadcasts** their public key, for example by attaching it to emails or posting it on a public forum. Anyone can then download it.

```mermaid
flowchart LR
    A["User A"] -- "PUa" --> B["User B"]
    A -- "PUa" --> C["User C"]
    A -- "PUa" --> D["Public forum"]
    B -- "PUb" --> A
```

- **Advantages**: very simple, and no third party is needed
- **Weakness: Forgery**:
  - An attacker can create their own key pair and announce it **in someone else's name**
  - Anyone who believes it sends messages the attacker can read
  - Until the real owner notices and warns people, the forgery works
  - Users can reduce the risk by checking a key's **fingerprint** (a short hash of it) with the owner directly

#### **2. Publicly Available Directory**

A **trusted third party** maintains a directory of public keys.

```mermaid
flowchart LR
    A["User A"] -- "Register PUa<br>(securely, e.g. in person)" --> DIR[("Public key directory<br>(trusted authority)")]
    B["User B"] -- "Register PUb" --> DIR
    DIR -- "Look up PUb" --> A
    DIR -- "Look up PUa" --> B
```

- **Requirements**:
  - The authority keeps a directory listing each **owner's name and public key**
  - Users **register** their key in person, or through another secure, authenticated method
  - Users can **replace** their key at any time, e.g. if it's been compromised, and old keys are removed
  - Users can access the directory **electronically**, over a secure, authenticated connection

- **Advantages**: much harder to forge than public announcement
- **Weakness**: the directory is a **single target**. An attacker who compromises it, or steals the authority's private key, can hand out **fake keys** for anyone

#### **3. Public-Key Authority**

The directory becomes an **active authority** with its own key pair. Every user knows the authority's **public key** $PU_{auth}$, and only the authority knows its **private key** $PR_{auth}$. Replies from the authority are **signed**, so users know they're genuine.

```mermaid
sequenceDiagram
    participant A as User A
    participant P as Public-key authority
    participant B as User B
    A->>P: 1. Request for PUb, T1
    P->>A: 2. E(PRauth, [PUb, Request, T1])
    A->>B: 3. E(PUb, [IDa, N1])
    B->>P: 4. Request for PUa, T2
    P->>B: 5. E(PRauth, [PUa, Request, T2])
    B->>A: 6. E(PUa, [N1, N2])
    A->>B: 7. E(PUb, N2)
```

| Step | Message | Purpose |
|------|---------|---------|
| 1 | Request $\parallel T_1$ | A asks for B's key, with a **timestamp** $T_1$ |
| 2 | $E(PR_{auth}, [PU_b \parallel \text{Request} \parallel T_1])$ | Signed by the authority, so A knows it's genuine. The returned request shows nothing was changed, and $T_1$ shows the reply is **new**, not a replay |
| 3 | $E(PU_b, [ID_A \parallel N_1])$ | A contacts B, with a nonce $N_1$ |
| 4 and 5 | Same as steps 1 and 2 | B gets A's public key from the authority |
| 6 | $E(PU_a, [N_1 \parallel N_2])$ | B returns $N_1$, proving it's really B, and adds $N_2$ |
| 7 | $E(PU_b, N_2)$ | A returns $N_2$, proving it's really A |

- Steps 1 to 5 **distribute the keys**, and steps 6 and 7 **authenticate** both users
- A and B can **store** each other's keys and reuse them later, without asking the authority again

- **Weaknesses**:
  - The authority must be **online** and contacted for every new key, so it becomes a **bottleneck** on large networks
  - The authority's stored keys are still a target

#### **4. Public-Key Certificates**

**Certificates** let users exchange keys **without contacting an authority each time**. A trusted **Certification Authority (CA)** checks a user's identity once, then gives them a **signed certificate** linking their identity to their public key. Anyone who trusts the CA can check the certificate using the **CA's public key**.

```mermaid
sequenceDiagram
    participant A as User A
    participant CA as Certification Authority
    participant B as User B
    A->>CA: Request certificate for PUa (proves identity)
    CA->>A: Cert A = E(PRauth, [T, IDa, PUa])
    B->>CA: Request certificate for PUb
    CA->>B: Cert B = E(PRauth, [T, IDb, PUb])
    A->>B: Cert A
    B->>A: Cert B
    Note over A,B: Each checks the other's certificate with PUauth, without contacting the CA again
```

- **Requirements for Certificates**:
  - Anyone can **read** a certificate to find the owner's name and public key
  - Anyone can **verify** that the certificate came from the CA and isn't forged
  - Only the **CA** can create and update certificates
  - Anyone can check that a certificate is **current**: not expired or revoked

**Issuing a certificate.** The user proves their identity to the CA, in person or by another authenticated method. The CA then creates and signs the certificate:

$$C_A = E(PR_{auth}, [T \parallel ID_A \parallel PU_a])$$

- $T$: the **timestamp**, when the certificate was issued
- $ID_A$: the owner's identity
- $PU_a$: the owner's public key

**Checking a certificate.** The receiver uses the CA's public key:

$$D(PU_{auth}, C_A) = D(PU_{auth}, E(PR_{auth}, [T \parallel ID_A \parallel PU_a])) = (T \parallel ID_A \parallel PU_a)$$

- Only the CA could have created something that $PU_{auth}$ unlocks, so the receiver knows the certificate is **genuine**
- The timestamp works like an **expiry date** on a credit card: old certificates are rejected, which protects against a stolen private key whose old certificate is still being used

- **Advantages**:
  - Certificates can be checked **offline**, with no need to contact the CA each time
  - The CA is no longer a bottleneck
  - The standard certificate format is **X.509**, covered in the next lesson

<br>

***

### **X.509 Certificates**

**X.509** isn't a technique on its own. It's a **standard**, part of the **X.500** series for **directory services**. It defines how a directory can provide **authentication services** to users, including how **public-key certificates** are stored and formatted. X.509 is the **accepted standard** for public-key certificates, and it's what HTTPS certificates use today.

#### **How a Certificate Is Signed and Checked**

```mermaid
flowchart LR
    subgraph CREATE["CA creates the certificate"]
        direction TB
        U["Unsigned certificate:<br>user ID, user's public key, ..."] --> H1["Hash"] --> S["Sign hash with<br>CA's private key"]
        U --> C["Signed certificate"]
        S --> C
    end
    subgraph CHECK["Anyone checks it"]
        direction TB
        C2["Signed certificate"] --> H2["Hash the<br>certificate fields"] --> CMP{"Same?"}
        C2 --> V["Check signature with<br>CA's public key"] --> CMP
        CMP -- "Yes" --> OK["Certificate is genuine"]
    end
    CREATE --> CHECK
```

- **Creating**:
  - The certificate fields (user ID, public key, CA details and so on) are **hashed**
  - The CA **signs the hash** with its **private key**
  - The signature is attached to the certificate

- **Checking**:
  - The receiver **hashes the certificate fields** themselves
  - They use the **CA's public key** on the signature to get the CA's hash
  - If the two hashes **match**, the certificate is genuine and unchanged

#### **X.509 Certificate Format**

| Field | Description | Version |
|-------|-------------|---------|
| **Version** | The certificate format version: 1 (default), 2 or 3. Each version includes all the fields of earlier versions, plus more | 1 |
| **Serial number** | A number, unique among certificates issued by this CA | 1 |
| **Signature algorithm identifier** | The algorithm and parameters used to **sign** the certificate | 1 |
| **Issuer name** | The name of the **CA** that created and signed the certificate | 1 |
| **Period of validity** | Two dates: **not before** and **not after**. The certificate is only valid between them | 1 |
| **Subject name** | The name of the **owner** of the public key | 1 |
| **Subject's public-key information** | The owner's **public key**, the algorithm it's used with (e.g. RSA), and any parameters | 1 |
| **Issuer unique identifier** | Optional bit string identifying the CA, in case the name was reused | 2 |
| **Subject unique identifier** | Optional bit string identifying the owner, in case the name was reused | 2 |
| **Extensions** | Extra fields, such as other names for the subject and what the key may be used for | 3 |
| **Signature** | A hash of all the other fields, signed with the **CA's private key**, plus the signature algorithm identifier | All |

**Version 3** is the current version.

- **Properties of Signed Certificates**:
  - **Only the CA** can create or change a certificate without it being detected
  - Certificates can be kept in **any public directory**, with no special protection, because any tampering breaks the signature
  - Each user does need a **trusted copy of the CA's public key**, obtained securely

#### **Certificate Revocation**

A certificate normally stays valid until its end date, but sometimes it must be **cancelled early**:

- The owner's **private key** has been compromised
- The owner is **no longer certified** by the CA, e.g. they left the organisation
- The **CA itself** has been compromised

The CA publishes a **Certificate Revocation List (CRL)**: a **signed** list of the certificates it has cancelled. Users should check the CRL before trusting a certificate.

| CRL Field | Description |
|-----------|-------------|
| Signature algorithm identifier | The algorithm used to sign the CRL |
| Issuer name | The CA that issued the CRL |
| This update date | When this CRL was issued |
| Next update date | When the next CRL will be issued |
| Revoked certificate entries | For each revoked certificate: its **serial number** and the **revocation date** |
| Signature | The whole CRL, signed with the CA's private key |

Revocation marks the certificate as invalid **from that point on**. It doesn't undo messages that were already sent. Today, many systems also use the **Online Certificate Status Protocol (OCSP)** to check a single certificate's status without downloading the whole list.

#### **Multiple CAs**

With a large number of users, one CA isn't practical. **Several CAs** share the load, and a compromised CA only affects **its own** users. But this raises a problem:

> User A trusts CA $X_1$. User B's certificate is signed by a different CA, $X_2$. How can A trust B's certificate without a trusted copy of $X_2$'s public key?

#### **Certificate Chains**

The CAs **certify each other**. $X_1$ signs a certificate for $X_2$'s public key, so anyone who trusts $X_1$ can now trust $X_2$.

**Notation:** $Y\langle\langle X \rangle\rangle$ means "the certificate of $X$, issued and signed by CA $Y$".

```mermaid
flowchart TB
    X1["CA X1<br>(trusted by A)"] -- "issues X1⟨⟨X2⟩⟩" --> X2["CA X2"]
    X2 -- "issues X2⟨⟨B⟩⟩" --> B["User B's certificate"]
    A["User A<br>(has X1's public key)"] -. "1. checks X1⟨⟨X2⟩⟩ with X1's key<br>→ now trusts X2's key" .-> X2
    A -. "2. checks X2⟨⟨B⟩⟩ with X2's key<br>→ now trusts B's key" .-> B
```

- **Steps**:
  - Step 1: $X_1$ and $X_2$ securely exchange public keys, and $X_1$ issues a certificate for $X_2$
  - Step 2: A gets $X_1\langle\langle X_2 \rangle\rangle$ and checks it with $X_1$'s public key, which A already trusts. Now A has a **trusted copy of $X_2$'s public key**
  - Step 3: A gets $X_2\langle\langle B \rangle\rangle$ and checks it with $X_2$'s public key. Now A trusts **B's public key**

The chain is written as:

$$X_1\langle\langle X_2 \rangle\rangle\ X_2\langle\langle B \rangle\rangle$$

Chains can be any length, as long as each certificate is signed by the CA before it:

$$X_1\langle\langle X_2 \rangle\rangle\ X_2\langle\langle X_3 \rangle\rangle \cdots X_N\langle\langle B \rangle\rangle$$

#### **Forward and Reverse Certificates**

In a hierarchy of CAs, each CA $X$ stores two kinds of certificates in its directory entry:

| Type | Certificate Of | Issued By | Example for CA $X$ |
|------|----------------|-----------|--------------------|
| **Forward certificate** | $X$ | **Other** CAs | $Y\langle\langle X \rangle\rangle$ |
| **Reverse certificate** | **Other** CAs | $X$ | $X\langle\langle Y \rangle\rangle$ |

Together, these let users build a chain from their own CA to any other CA.

#### **X.509 in Practice**

- **HTTPS** uses X.509 version 3 certificates
- Browsers and operating systems come with a built-in list of trusted **root CAs**
- A website's certificate is usually signed by an **intermediate CA**, which is signed by a root CA
- The browser checks the whole chain, $\text{Root}\langle\langle \text{Intermediate} \rangle\rangle\ \text{Intermediate}\langle\langle \text{Website} \rangle\rangle$, plus the validity dates and revocation status

<br>

***

### **Authentication Protocol Examples**

A **protocol** is a set of rules that every party must follow. **Authentication protocols** use security and cryptographic rules so that the parties can **prove their identities** to each other before they start communicating.

- **Two Approaches**:
  - **One-way authentication**: only one party proves its identity to the other
  - **Mutual (two-way) authentication**: **both** parties prove their identities to each other

**Notation:** $\{X\}_K$ means $X$ encrypted with key $K$, and $A \rightarrow B : X$ means A sends $X$ to B.

#### **Nonces**

A **nonce** ("**n**umber used **once**") is a value used **only once** in a protocol run, then thrown away when the session ends.

- **What Makes a Good Nonce**:
  - **Unique**: it must never repeat. A random number, a counter or a timestamp can all work
  - **Unpredictable**: when it's used as a **challenge**, it should be random, so an attacker can't prepare an answer in advance

- **Stopping Replay Attacks**:
  - In a **replay attack**, an attacker captures a valid message and **sends it again later**, pretending to be the original sender
  - A message containing a new nonce **can't be reused**: the receiver spots the old nonce and rejects it
  - With a **timestamp** nonce, the receiver can reject messages that are too old

- **Challenge-Response**: a nonce can also be used as a **challenge**. Here the server checks that the client has the shared key $K_s$:

```mermaid
sequenceDiagram
    participant C as Client
    participant S as Server
    Note over C,S: Both share a secret key Ks
    S->>C: Ns (random challenge)
    C->>S: {Ns}Ks
    Note over S: Decrypts. If it matches Ns, the client has Ks
```

$$\text{Server} \rightarrow \text{Client} : N_s$$

$$\text{Client} \rightarrow \text{Server} : \{N_s\}_{K_s}$$

#### **Andrew Secure RPC Protocol**

A **Remote Procedure Call (RPC)** lets a program **run code on another system**: on the same network, or over the internet. The system asking to run code must be **authenticated** and **authorised** first.

The **Andrew Secure RPC** protocol lets a client A and a server B, who share a secret key $K_{AB}$, authenticate each other and set up a **new session key** $K'_{AB}$. It has **4 messages**: the first 3 are a handshake to authenticate both sides, and the 4th sends the new session key.

```mermaid
sequenceDiagram
    participant A as Client A
    participant B as Server B
    A->>B: 1. A, {NA}Kab
    B->>A: 2. {NA + 1, NB}Kab
    A->>B: 3. {NB + 1}Kab
    B->>A: 4. {K'ab, N'B}Kab
    Note over A: Nothing in message 4 links it to NA,<br>so an old message 4 can be replayed
```

| Step | Message | Purpose |
|------|---------|---------|
| 1 | $A \rightarrow B : A, \{N_A\}_{K_{AB}}$ | A sends a nonce |
| 2 | $B \rightarrow A : \{N_A + 1, N_B\}_{K_{AB}}$ | B proves it has $K_{AB}$ by returning $N_A + 1$, and sends its own nonce |
| 3 | $A \rightarrow B : \{N_B + 1\}_{K_{AB}}$ | A proves it has $K_{AB}$ |
| 4 | $B \rightarrow A : \{K'_{AB}, N'_B\}_{K_{AB}}$ | B sends the new session key |

- **Weaknesses**:
  - **Message 4 isn't linked to this run**: it contains nothing A sent (no $N_A$), so A can't tell whether it's new
  - An attacker can **replay an old message 4**, making A use an **old session key**, which may have been compromised
  - **No integrity check**, so messages can be modified without being detected

**Improved version.** The new key is sent **together with $N_A$**, and every message has a **MAC** for integrity:

```mermaid
sequenceDiagram
    participant A as Client A
    participant B as Server B
    A->>B: 1. A, {NA}Kab, MAC
    B->>A: 2. {NA, NB, K'ab, B}Kab, MAC
    A->>B: 3. {A, NB + 1}Kab, MAC
    Note over A,B: New key K'ab is sent with NA, so A knows it's fresh
```

$$A \rightarrow B : A, \{N_A\}_{K_{AB}}, \text{MAC}_{K_{AB}}(A, \{N_A\}_{K_{AB}})$$

$$B \rightarrow A : \{N_A, N_B, K'_{AB}, B\}_{K_{AB}}, \text{MAC}_{K_{AB}}(\{N_A, N_B, K'_{AB}, B\}_{K_{AB}})$$

$$A \rightarrow B : \{A, N_B + 1\}_{K_{AB}}, \text{MAC}_{K_{AB}}(\{A, N_B + 1\}_{K_{AB}})$$

- **What the Improvements Do**:
  - $K'_{AB}$ arrives **with $N_A$**, so A knows the key is **new**
  - Including identities ($A$, $B$) stops messages from being reused in the wrong direction
  - **MACs** detect any **modification**
  - Both sides prove they have $K_{AB}$, giving **mutual authentication**

#### **Needham-Schroeder Protocol**

The **Needham-Schroeder** protocol (1978) has influenced many later protocols, including **Kerberos**. Unlike Andrew RPC, it uses a **Trusted Third Party (TTP)**: a server $S$ that both A and B trust. S already shares a key $K_{AS}$ with A and a key $K_{BS}$ with B, and it creates a new session key $K$ for A and B.

```mermaid
sequenceDiagram
    participant A as A
    participant S as Server S
    participant B as B
    A->>S: 1. A, B, NA
    S->>A: 2. {NA, B, K, {K, A}Kbs}Kas
    A->>B: 3. {K, A}Kbs
    B->>A: 4. {NB}K
    A->>B: 5. {NB - 1}K
```

| Step | Message | Purpose |
|------|---------|---------|
| 1 | $A \rightarrow S : A, B, N_A$ | A asks S for a key to talk to B |
| 2 | $S \rightarrow A : \{N_A, B, K, \{K, A\}_{K_{BS}}\}_{K_{AS}}$ | S sends $K$ to A, plus a **ticket** for B. $N_A$ shows A the reply is new |
| 3 | $A \rightarrow B : \{K, A\}_{K_{BS}}$ | A forwards the ticket. Only B can read it |
| 4 | $B \rightarrow A : \{N_B\}_K$ | B challenges A |
| 5 | $A \rightarrow B : \{N_B - 1\}_K$ | A proves it has $K$ |

- **The Flaw (Denning-Sacco Attack)**:
  - Message 3 contains **nothing new for B**, so B **can't tell if the ticket is new or old**
  - If an attacker ever learns an **old session key** $K$, they can **replay** the old message 3 to B
  - The attacker can then answer B's challenge in steps 4 and 5, since they know $K$, and **impersonate A**

**Improved version.** B adds its **own nonce** $N_{B1}$ to the ticket at the start:

```mermaid
sequenceDiagram
    participant A as A
    participant S as Server S
    participant B as B
    A->>B: 1. A
    B->>A: 2. {A, NB1}Kbs
    A->>S: 3. A, B, NA, {A, NB1}Kbs
    S->>A: 4. {NA, B, K, {K, NB1, A}Kbs}Kas
    A->>B: 5. {K, NB1, A}Kbs
    B->>A: 6. {NB}K
    A->>B: 7. {NB - 1}K
    Note over B: NB1 in message 5 proves the ticket is new
```

| Step | Message |
|------|---------|
| 1 | $A \rightarrow B : A$ |
| 2 | $B \rightarrow A : \{A, N_{B1}\}_{K_{BS}}$ |
| 3 | $A \rightarrow S : A, B, N_A, \{A, N_{B1}\}_{K_{BS}}$ |
| 4 | $S \rightarrow A : \{N_A, B, K, \{K, N_{B1}, A\}_{K_{BS}}\}_{K_{AS}}$ |
| 5 | $A \rightarrow B : \{K, N_{B1}, A\}_{K_{BS}}$ |
| 6 | $B \rightarrow A : \{N_B\}_K$ |
| 7 | $A \rightarrow B : \{N_B - 1\}_K$ |

- **Why It Works**:
  - B creates $N_{B1}$ at the **start** of this run
  - S puts $N_{B1}$ **inside the ticket**, which only S and B can create or read
  - When B sees its own $N_{B1}$ in message 5, it knows the ticket was made **for this run**, so an old ticket can't be replayed

#### **Comparison**

| Protocol | Third Party | Main Weakness | Fix |
|----------|-------------|---------------|-----|
| Andrew Secure RPC | No | Session key in message 4 isn't linked to $N_A$, so it can be replayed | Send $K'_{AB}$ with $N_A$, add MACs |
| Needham-Schroeder | Yes (server $S$) | B can't tell if the ticket in message 3 is new | Add B's nonce $N_{B1}$ to the ticket (or timestamps, as in Kerberos) |

<br>

***

### **SSL, Kerberos, RADIUS and SAML**

This lesson looks at four **real-world authentication protocols** and what each is used for.

| Protocol | Used For | Main Cryptography |
|----------|----------|-------------------|
| **SSL / TLS** | Securing web traffic (HTTPS) and other connections | Certificates, public keys, then symmetric keys |
| **Kerberos** | Single sign-on inside an organisation's network | Symmetric keys and tickets |
| **RADIUS** | Logging in to network access: dial-up, Wi-Fi, VPN | Shared secrets between servers |
| **SAML** | Single sign-on across websites and organisations | Signed XML assertions |

#### **Secure Sockets Layer (SSL) and TLS**

**SSL** was the original protocol for securing internet traffic. It has been replaced by **Transport Layer Security (TLS)**, which supports newer algorithms such as **AES** and **HMAC** (keyed hash functions). SSL/TLS sits **between the transport layer (TCP) and the application layer**, so it can protect HTTP and many other protocols. **HTTPS** is HTTP running over TLS.

| Version | Status |
|---------|--------|
| SSL 2.0 and 3.0 | **Insecure**, banned |
| TLS 1.0 and 1.1 | **Deprecated** |
| TLS 1.2 | Secure, widely used |
| TLS 1.3 (2018) | Current version: faster and simpler |

- **What SSL/TLS Provides**:
  - **Authentication**: using public-key cryptography, digital signatures and **certificates**
  - **Confidentiality**: data is **encrypted**, so only the intended recipient can read it
  - **Integrity**: every message has a **MAC**, so any tampering is **detected**

#### **The SSL Handshake (Simplified)**

Before sending data, the client and server perform a **handshake** to authenticate and agree on a session key.

```mermaid
sequenceDiagram
    participant C as Client
    participant S as Server
    C->>S: 1. Hello, supported ciphers, N_client
    S->>C: 2. Certificate, chosen cipher, N_server
    Note over C: Checks the certificate, gets the server's public key.<br>Picks pre-master secret S. Computes K from S, N_client, N_server
    C->>S: 3. {S} encrypted with server's public key, MAC of handshake using K
    Note over S: Decrypts S with its private key, computes the same K
    S->>C: 4. MAC of handshake using K
    Note over C,S: 5. Data encrypted with session key K
```

| Step | Message | Purpose |
|------|---------|---------|
| 1 | Client → Server: SSL request, cipher list, $N_{Client}$ | The client says which ciphers it supports and sends a nonce |
| 2 | Server → Client: $\text{Certificate}_{Server}$, cipher, $N_{Server}$ | The server sends its certificate, picks a cipher, and sends its own nonce |
| 3 | Client → Server: $\{S\}_{+K_{Server}},\ H(\text{msgs}, \text{CLIENT}, K)$ | The client **checks the certificate** with the CA's public key, picks a random **pre-master secret** $S$, and encrypts it with the server's public key. Both sides compute the **session key** $K$ from $S$, $N_{Client}$ and $N_{Server}$. The keyed hash proves the client has $K$ |
| 4 | Server → Client: $H(\text{msgs}, \text{SERVER}, K)$ | The server decrypts $S$ with its private key, computes the same $K$, and proves it has it |
| 5 | Client ↔ Server: data protected with $K$ | All data is encrypted and MACed with the session key |

- **About the Keyed Hashes**:
  - $H(\ldots, K)$ is a **MAC over all the handshake messages**, computed **using** $K$. The key $K$ itself is **never sent**
  - It proves both sides computed the **same $K$**, and that nobody **tampered** with the handshake

- **Who Is Authenticated?**:
  - Usually **only the server** is authenticated (one-way), through its certificate
  - **Client authentication** is optional: the server asks for the client's certificate, and the client **signs** the handshake with its private key to prove it owns that certificate
  - Mutual authentication is used in high-security settings, e.g. between company systems

- **Disadvantages**:
  - **Certificate checking takes time**, especially checking that a certificate **hasn't been revoked**
  - With mutual authentication, this has to be done for both sides

- **TLS 1.3**: instead of the client encrypting $S$ with the server's RSA key, both sides use **ephemeral Diffie-Hellman** to agree on $K$. This gives **forward secrecy**: stealing the server's private key later doesn't expose past sessions

#### **Kerberos**

**Kerberos**, designed at **MIT**, uses **symmetric-key cryptography** to authenticate users and services on a network. It's the default login system in **Windows Active Directory** networks, and provides **Single Sign-On (SSO)**: the user logs in **once** and can then use many services without typing their password again.

**The problem it solves:** without Kerberos, a user accessing the mail server, a database server and a web server would have to log in to **each one separately**. That's tiresome, and it means sending the password many times.

- **Components**:
  - **Key Distribution Center (KDC)**: the trusted third party, made up of two parts:
    - **Authentication Server (AS)**: checks the user's login and issues a **Ticket Granting Ticket (TGT)**
    - **Ticket Granting Server (TGS)**: takes the TGT and issues **service tickets** for specific services
  - The KDC shares a **secret key with every user and service**, so $N$ users need $N$ keys
  - The KDC also has its **own key**, never shared, which it uses to encrypt TGTs
  - **Tickets**: encrypted messages containing a **session key** and the **user's identity**. They prove to a service that the KDC has authenticated the user

```mermaid
sequenceDiagram
    participant U as User
    participant AS as Authentication Server
    participant TGS as Ticket Granting Server
    participant V as Service
    Note over AS,TGS: Together these form the KDC
    U->>AS: 1. Login request
    AS->>U: 2. {Session key SA, TGT}KA (KA comes from the user's password)
    Note over U: Decrypts with KA once, then forgets the password
    U->>TGS: 3. TGT, {IDuser, timestamp}SA, service wanted
    TGS->>U: 4. {ServiceID, KUS, ticket = {IDuser, KUS}Kservice}SA
    U->>V: 5. Ticket, {IDuser, timestamp}KUS
    V->>U: 6. Access granted
```

**Stage 1: Login (once per day or session)**

- The user logs in, and the AS sends back a **session key** $S_A$ and a **TGT**
- The reply is encrypted with the user's key $K_A$, which is **derived from their password**
- The user's computer decrypts it, then **forgets** $K_A$ and the password
- From now on, it uses $S_A$ to talk to the KDC

**Stage 2: Getting access to a service (once per service)**

$$\text{REQUEST} = (\text{TGT}, \{ID_{user}, \text{TimeStamp}\}_{S_A})$$

$$\text{REPLY} = \{\text{ServiceID}, K_{US}, \{ID_{user}, K_{US}\}_{K_{Service}}\}_{S_A}$$

- The user sends the **TGT** and an **authenticator** (their ID and the time, encrypted with $S_A$)
- The TGS replies with a **session key** $K_{US}$ for the user and the service, and a **service ticket** $\{ID_{user}, K_{US}\}_{K_{Service}}$, all encrypted with $S_A$
- The user gives the service ticket to the service, which decrypts it with its own key $K_{Service}$ and learns $K_{US}$
- Each service ticket works for **one service only**. Other services need their own tickets, which the user's computer fetches automatically

- **Limitations**:
  - **Single point of failure**: if the KDC goes down, **nobody can log in** to anything
  - **Attractive target**: the KDC holds **every user's key**, so compromising it compromises everyone
  - **Clocks must be in sync**: tickets and authenticators carry **timestamps**, and a clock difference of a few minutes causes them to be rejected

#### **RADIUS**

**Remote Authentication Dial-In User Service (RADIUS)** is a **client-server** protocol, originally designed to authenticate users of **dial-in** servers. Today it's mostly used for **enterprise Wi-Fi** (WPA2/WPA3-Enterprise) and **VPN** logins. It runs over **UDP**, a connectionless protocol.

- **Roles**:
  - **Network Access Server (NAS)**: the device the user connects to, such as a dial-in server, Wi-Fi access point or VPN gateway. It acts as the **RADIUS client**
  - **RADIUS server**: checks the credentials and says **yes or no**
  - A RADIUS server can also act as a **proxy**, forwarding requests to other RADIUS or authentication servers

```mermaid
sequenceDiagram
    participant U as User
    participant N as NAS (RADIUS client)
    participant R as RADIUS server
    U->>N: 1. Connects (e.g. PPP, Wi-Fi, VPN)
    N->>U: 2. Asks for username and password
    U->>N: 3. Credentials (PAP or CHAP)
    N->>R: 4. Access-Request (over UDP)
    R->>N: 5. Access-Accept, Access-Reject or Access-Challenge
    N->>U: 6. Grants access, rejects, or asks a further challenge
```

- **Steps**:
  - Step 1: The user connects to the NAS, e.g. with a **Point-to-Point Protocol (PPP)** connection
  - Step 2: The NAS asks for a username and password
  - Step 3: The user sends credentials using **PAP** (password sent as-is) or **CHAP** (challenge-response, so the password isn't sent)
  - Step 4: The NAS sends an **Access-Request** to the RADIUS server
  - Step 5: The server replies **Access-Accept**, **Access-Reject**, or **Access-Challenge** (asking for more information)
  - Step 6: The NAS grants access, rejects the user, or passes on the challenge

- **Notes**:
  - This is the **three-party model** from lesson 9.1: the NAS is the responder, and the RADIUS server is the authenticator
  - RADIUS also handles **accounting**: recording usage, e.g. for billing
  - Only the **password** in a RADIUS message is hidden. The rest of the message isn't encrypted

#### **SAML**

**Security Assertion Markup Language (SAML)** is an **open standard** for **single sign-on across websites**, even across **different organisations** that have an agreement with each other. Messages are written in **XML**.

- **Roles**:
  - **Identity Provider (IdP)**: where the user actually logs in, e.g. a university login page
  - **Service Provider (SP)**: the website or service the user wants to use, e.g. a library database
  - The IdP sends the SP a **signed assertion**: a statement saying who the user is

```mermaid
sequenceDiagram
    participant U as User's browser
    participant SP as Service Provider (SP)
    participant IdP as Identity Provider (IdP)
    U->>SP: 1. Tries to open the service
    SP->>U: 2. Redirect to the IdP with a SAML request
    U->>IdP: 3. SAML request
    Note over U,IdP: 4. User logs in (only if not already logged in)
    IdP->>U: 5. Signed SAML assertion
    U->>SP: 6. Signed SAML assertion
    Note over SP: 7. Checks the IdP's signature, grants access
```

| Assertion Type | Contains |
|----------------|----------|
| **Authentication assertion** | Who the user is, **how** they logged in (e.g. password, MFA), and **when** |
| **Attribute assertion** | Facts about the user, e.g. role or department, which the SP uses to decide access |
| **Authorisation decision assertion** | Whether the user is **allowed or denied** access to a specific resource |

- **Why SAML Is Useful**:
  - The user logs in **once** at the IdP and can use many SPs without logging in again
  - SPs **never see the user's password**, only the signed assertion
  - Widely used for logging in to business cloud apps through a company or university account
  - Newer web and mobile apps often use **OpenID Connect** for the same purpose

<br>

***